<a href="https://colab.research.google.com/github/Balkrishnakc17/Numerical-Methods/blob/main/Linear-Spline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Linear Spline Interpolation Project

Linear spline interpolation is a method of fitting curves using linear polynomials (straight line segments) to construct new data points within the range of a discrete set of known data points.

### Mathematical Concept
Given a set of $n+1$ points $(x_0, y_0), (x_1, y_1), \dots, (x_n, y_n)$ sorted such that $x_0 < x_1 < \dots < x_n$, the linear spline $S(x)$ is defined as a piecewise linear function:

$$S(x) = S_i(x) = y_i + \frac{y_{i+1} - y_i}{x_{i+1} - x_i} (x - x_i) \quad \text{for } x \in [x_i, x_{i+1}]$$

Where:
* $S_i(x)$ is the linear equation representing the line segment connecting interval $[x_i, x_{i+1}]$.
* The term $\frac{y_{i+1} - y_i}{x_{i+1} - x_i}$ represents the slope ($m_i$) of that specific interval.

## Part 1: Basic Linear Spline Demonstration

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Define a simple, sample dataset
x_data = np.array([0.0, 1.5, 3.0, 4.5, 6.0])
y_data = np.array([1.0, 3.5, 2.0, 5.0, 4.0])

# Basic linear spline function from scratch
def basic_linear_spline(x, x_pts, y_pts):
    """
    Evaluates the linear spline at a given point x using provided data points.
    """
    # Clip x to stay within the domain boundary
    if x < x_pts[0] or x > x_pts[-1]:
        raise ValueError("x is outside the interpolation range.")

    # Find the interval containing x
    for i in range(len(x_pts) - 1):
        if x_pts[i] <= x <= x_pts[i+1]:
            # Apply linear spline formula
            slope = (y_pts[i+1] - y_pts[i]) / (x_pts[i+1] - x_pts[i])
            return y_pts[i] + slope * (x - x_pts[i])

# Vectorize the function to support array operations easily
vectorized_spline = np.vectorize(basic_linear_spline, excluded=['x_pts', 'y_pts'])

# Generate a dense set of x values to plot the spline smoothly
x_dense = np.linspace(x_data[0], x_data[-1], 200)
y_spline = vectorized_spline(x_dense, x_data, y_data)

# Plotting the baseline setup
plt.figure(figsize=(10, 5))
plt.plot(x_dense, y_spline, 'g-', label='Linear Spline S(x)', linewidth=2)
plt.scatter(x_data, y_data, color='red', s=80, zorder=5, label='Data Points')
plt.title('Basic Linear Spline Interpolation')
plt.xlabel('X')
plt.ylabel('Y')
plt.legend()
plt.grid(True)
plt.show()

IndexError: invalid index to scalar variable.

## Part 2: Interactive Linear Spline with Error Analysis

Ordinary linear interpolation works, but it can be a "black box" when debugging or trying to understand which interval is active.

### Why this interactive version is a creative and useful improvement:
1. **Dynamic Visual Feedback**: Using `ipywidgets` allows real-time exploration, helping build intuition.
2. **Active Segment Highlighting**: This programmatically identifies and colors the exact mathematical segment containing the chosen $x$, showcasing which formula partition is actively computing the result.
3. **Real-world Verification (Ground Truth)**: We introduce a true mathematical function ($f(x) = \sin(x) + \cos(x/2)$) as our source, allowing us to evaluate the exact **interpolation error** dynamically.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

# Define the Ground Truth (True Function)
def true_function(x):
    return np.sin(x) + np.cos(x / 1.5)

# Select 6 control nodes for our interpolation
x_nodes = np.array([0.0, 1.2, 2.8, 4.5, 6.2, 8.0])
y_nodes = true_function(x_nodes)

def interactive_spline(target_x):
    # 1. Calculate linear spline prediction dynamically
    # Find the active segment index i
    idx = 0
    for i in range(len(x_nodes) - 1):
        if x_nodes[i] <= target_x <= x_nodes[i+1]:
            idx = i
            break

    x_left, x_right = x_nodes[idx], x_nodes[idx+1]
    y_left, y_right = y_nodes[idx], y_nodes[idx+1]

    # Calculate slope and estimate point value
    slope = (y_right - y_left) / (x_right - x_left)
    predicted_y = y_left + slope * (target_x - x_left)

    # 2. Compute error metrics
    actual_y = true_function(target_x)
    absolute_error = abs(actual_y - predicted_y)

    # 3. Plotting everything dynamically
    plt.figure(figsize=(12, 6.5))

    # Draw dense ground truth function
    x_dense = np.linspace(x_nodes[0], x_nodes[-1], 300)
    plt.plot(x_dense, true_function(x_dense), 'k--', alpha=0.6, label='True Function $f(x)$')

    # Plot all base linear spline lines
    plt.plot(x_nodes, y_nodes, color='lightgray', linestyle='-', label='Other Spline Segments', zorder=1)

    # HIGHLIGHT: Color the active segment actively performing the interpolation
    plt.plot([x_left, x_right], [y_left, y_right], color='magenta', linewidth=4, label='Active Segment $S_i(x)$', zorder=2)

    # Draw control nodes
    plt.scatter(x_nodes, y_nodes, color='blue', s=70, label='Data Nodes', zorder=3)

    # Mark the user prediction point and the actual point on the true curve
    plt.scatter(target_x, predicted_y, color='red', marker='o', s=120, label='Interpolated Point', zorder=4)
    plt.scatter(target_x, actual_y, color='green', marker='x', s=120, label='Actual Point', zorder=4)

    # Draw a line between predicted and actual point illustrating the error
    plt.vlines(target_x, min(predicted_y, actual_y), max(predicted_y, actual_y),
               colors='red', linestyles='dotted', linewidth=2.5, label='Error Gap')

    # Titles and formatting
    plt.title('Creative Interactive Spline Interpolator with Error Visualization', fontsize=14)
    plt.xlabel('X', fontsize=12)
    plt.ylabel('Y', fontsize=12)
    plt.xlim(x_nodes[0] - 0.5, x_nodes[-1] + 0.5)
    plt.ylim(min(true_function(x_dense)) - 0.5, max(true_function(x_dense)) + 0.5)
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)

    # Display calculations directly as text in the visualization area
    text_info = (
        f"Active Segment: [{x_left:.2f}, {x_right:.2f}]\n"
        f"Slope (m): {slope:.4f}\n"
        f"Interpolated S({target_x:.2f}): {predicted_y:.4f}\n"
        f"True Value f({target_x:.2f}): {actual_y:.4f}\n"
        f"Absolute Error: {absolute_error:.4f}"
    )
    plt.text(x_nodes[0], min(true_function(x_dense)) - 0.2, text_info,
             bbox=dict(facecolor='white', alpha=0.9, edgecolor='gray'), fontsize=11, family='monospace')
    plt.show()

# Create Colab slider to interact with x values seamlessly
interact(interactive_spline, target_x=widgets.FloatSlider(
    value=3.5,
    min=0.0,
    max=8.0,
    step=0.05,
    description='Select X:',
    continuous_update=True
));